2. In a new .ipynb/.py file called **uva_analysis**, analyze foreign giving to UVA. Compare UVA to appropriate peer institutions and the market overall. Propose a strategy for finding new sources of foreign money and provide a quantitative analysis.
 3. In a document file (.doc, .docx, .pdf, etc.), summarize your findings:
     1. Executive Summary: What are your main findings and recommendations, in a single paragraph?
     2. Summarize the results of your initial analyses of overall giving and giving to UVA (1-3 paragraphs).
     3. Describe a strategy for finding new potential sources of money for UVA. What countries or institutions should UVA target, and why? (1-3 paragraphs)
     4. Conclude with a summary of UVA's prospects for increasing foreign giving. (1 paragraph).

 In total, your submission repo contain submit at least .ipynb/.py analysis files and a document file with your report (.doc, .docx, .pdf, etc.). If you divide the work among members of a group, make sure that you understand the code being written and submitted by the other members of your group.

In [17]:
import pandas as pd
import numpy as np
df = pd.read_csv('/Users/colesherwin/Documents/understanding_uncertainty/uu_wrangling_and_eda/lab_1/ForeignGifts_edu.csv')
fixed_col_names = ['id', 'opeid', 'institution_name', 'city', 'state', 'foreign_gift_received_date', 
                   'foreign_gift_amount', 'gift_type', 'country_of_giftor', 'giftor_name']
df.columns = fixed_col_names
non_uva_df = df[df['institution_name'] != 'University of Virginia']
uva_df = df[df['institution_name'] == 'University of Virginia']
#uva_df.head(3).T
non_uva_df

,id,opeid,institution_name,city,state,foreign_gift_received_date,foreign_gift_amount,gift_type,country_of_giftor,giftor_name
0,1,102000,Jacksonville State University,Jacksonville,AL,43738,250000,Monetary Gift,CHINA,NaN
1,2,104700,Troy University,Troy,AL,43592,463657,Contract,CHINA,Confucius Institute Headquarters
2,3,105100,University of Alabama,Tuscaloosa,AL,43466,3649107,Contract,ENGLAND,Springer Nature Customer Service Ce
3,4,105100,University of Alabama,Tuscaloosa,AL,43472,1000,Contract,SAUDI ARABIA,Saudi Arabia Education Mission
4,5,105100,University of Alabama,Tuscaloosa,AL,43479,49476,Contract,SAUDI ARABIA,Saudi Arabia Education Mission
...,...,...,...,...,...,...,...,...,...,...
28216,28217,4279700,Albert Einstein College of Medicine,Bronx,NY,42704,381717,Contract,CHINA,Chia Tai TianQing Pharmaceutical Gr
28217,28218,4279700,Albert Einstein College of Medicine,Bronx,NY,42778,444938,Contract,ISRAEL,BL Oncology Ltd
28218,28219,4279700,Albert Einstein College of Medicine,Bronx,NY,42907,1064580,Contract,ENGLAND,Roche Products Limited
28219,28220,4279700,Albert Einstein College of Medicine,Bronx,NY,42948,737375,Contract,SWITZERLAND,F Hoffman-La Roche Ltd


In [18]:
import plotly.express as px

others = non_uva_df.groupby('institution_name')['foreign_gift_amount'].sum()
uva_total = uva_df['foreign_gift_amount'].sum()

fig3 = px.histogram(others, x='foreign_gift_amount', nbins=50,
                    labels={'foreign_gift_amount': 'Total foreign gifts per institution'})
fig3.add_vline(x=uva_total, line_dash='dash', line_color='orange',
               annotation_text=f'UVA: ${uva_total:,.0f}', annotation_position='top')
fig3.update_layout(showlegend=False)

In [19]:
others_mean = non_uva_df.groupby('institution_name')['foreign_gift_amount'].mean()
uva_mean = uva_df['foreign_gift_amount'].mean()

fig4 = px.histogram(others_mean, x='foreign_gift_amount', nbins=50,
                    labels={'foreign_gift_amount': 'Total foreign gifts per institution'})
fig4.add_vline(x=uva_mean, line_dash='dash', line_color='orange',
               annotation_text=f'UVA: ${uva_mean:,.0f}', annotation_position='top')
fig4.update_layout(showlegend=False)

In [39]:
others_combined_mean = non_uva_df['foreign_gift_amount'].mean()
uva_mean = uva_df['foreign_gift_amount'].mean()
combined = {'UVA': uva_mean, 'Others': others_combined_mean}
combined_df = pd.Series(combined, name='mean_gift').rename_axis('group').reset_index()
fig5 = px.bar(combined_df, x = 'group', y = 'mean_gift')
fig5.show()


In [ ]:
#run first time
#uva_df['foreign_gift_received_date'] = pd.to_datetime(
   #uva_df['foreign_gift_received_date'], unit='D', origin='1899-12-30'
#)
top_countries = (uva_df.groupby('country_of_giftor')['foreign_gift_amount']
                       .sum().nlargest(5).index)
top5 = uva_df[uva_df['country_of_giftor'].isin(top_countries)]
fig4 = px.scatter(top5, x = 'foreign_gift_received_date', y = 'foreign_gift_amount',
                  labels={'foreign_gift_received_date':'Date Received Foreign Gift',
                          'foreign_gift_amount':'Amount Received'}, 
                  color = 'country_of_giftor',
                  trendline = 'ols',
                  title = 'UVA Date of Gift vs Amount of Gift')
fig4.update_traces(selector=dict(mode='lines'), line_dash='dash')
fig4.show()

/Users/colesherwin/miniconda3/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2018: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr / self.centered_tss


In [40]:
totals = df.groupby('institution_name')['foreign_gift_amount'].sum()

peers = ['University of Virginia', 'College of William & Mary',
         'Virginia Polytechnic Institute & State University', 'George Mason University',
         'Virginia Commonwealth University', 'University of North Carolina - Chapel Hill',
         'University of Michigan - Ann Arbor', 'University of California, Berkeley',
         'Duke University', 'Georgetown University', 'Johns Hopkins University']

peer_df = totals.loc[peers].sort_values().reset_index()
peer_df['color'] = np.where(peer_df['institution_name'] == 'University of Virginia', 'orange', 'steelblue')

fig6 = px.bar(peer_df, x='foreign_gift_amount', y='institution_name',
              labels={'foreign_gift_amount': 'Total foreign funding ($)', 'institution_name': ''},
              title='UVA vs peer institutions, 2014–2020')
fig6.update_traces(marker_color=peer_df['color'])
fig6.update_layout(showlegend=False)
fig6.show()

In [ ]:
# used claude to help
market = df.groupby('country_of_giftor')['foreign_gift_amount'].sum().nlargest(10)
uva_c = uva_df.groupby('country_of_giftor')['foreign_gift_amount'].sum()

gap = pd.DataFrame({'All institutions': market,
                    'UVA': uva_c.reindex(market.index).fillna(0)}).reset_index()

fig7 = px.bar(gap, x='country_of_giftor', y=['All institutions', 'UVA'], barmode='group',
              log_y=True,
              labels={'country_of_giftor': '', 'value': 'Total ($, log scale)', 'variable': ''},
              title='Top 10 source countries: market vs UVA')
fig7.show()

# Campaign Plan

- In lou of UVA's below par funding from foreign aid. I beleive we should seek contracts within foreign countries. We could offer graduate level schooling, provide education sesions, and engage in research opportunities on behalf of other countries. The fee for return strategy works. As seen in China's belt and road initiative.
- The first opportunity would be to pick and choice what smaller/less-prestigous institutions projects we may be able to take over at the end of their contract. We could sort by similar schools, but I would propose sorting by what UVA's strengths are. A great example of this is the new pathway for masters students in Korea. This would allow korean masters students to travel to UVA to engage earn their masters degree here.

Agreements like this could result in an uptick in foreign funding. More deeply, it would allow for marketing in foreign countries. Potentially, causing an uptick in foreign student enrollment. 

- Next we could target our weaker neighbors/competitors in current contract agreements. This would be ODU, JMU, Virginia Polytechnic Institute & State University, George Mason, William and Mary, and VCU. Virginia ranks higher in endowment than others but not quite in foreign funding. However, UVA does carry a higher name weight than these.

In [ ]:
# 1 - 3 - 6
print(round(df.groupby('institution_name')['foreign_gift_amount'].sum().mean(), 2))
print(df.groupby('institution_name')['foreign_gift_amount'].sum().median())

52202878.97
6486791.5
